### Clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


data_df = pd.read_csv(
    'GSE33000_Top10000_Var.csv',
    index_col=0,
    engine='python'
)

X = data_df.iloc[:, :-1].values   # shape: (624, 10000)
y = data_df.iloc[:, -1].values    # disease labels

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# pca = PCA(n_components=500, random_state=42)
# X_reduced = pca.fit_transform(X_scaled)

som_x, som_y = 20, 15  # 300 neurons

som = MiniSom(
    som_x,
    som_y,
    X_scaled.shape[1],
    sigma=1.5,
    learning_rate=0.5,
    neighborhood_function='gaussian',
    random_seed=42
)

som.random_weights_init(X_scaled)

som.train_batch(
    X_scaled,
    num_iteration=10000,
    verbose=True
)

# Winner som neuron for each sample
winner_coords = np.array([som.winner(x) for x in X_scaled])

cluster_index = np.ravel_multi_index(
    winner_coords.T,
    (som_x, som_y)
)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# plt.figure(figsize=(8, 6))

# scatter = plt.scatter(
#     X_pca[:, 0],
#     X_pca[:, 1],
#     c=cluster_index,
#     cmap='RdYlGn',
#     alpha=0.7,
#     s=30
# )

# plt.xlabel('PC1')
# plt.ylabel('PC2')
# plt.title('PCA projection colored by SoM clusters')
# plt.colorbar(scatter, label='SOM cluster')
# plt.tight_layout()
# plt.show()


### CLASIFICATION

In [ ]:
from sklearn.preprocessing import scale

data = pd.read_csv('GSE33000_Top10000_Var.csv',index_col=0, engine='python')
diagnosis_map = {'AD': 1, 'HD': 1, 'C': 0}
data['Diagnosis'] = data['Diagnosis'].map(diagnosis_map)
labels = data['Diagnosis'].values
data = data[data.columns[:-1]]
data = scale(data.values)

In [ ]:
def classify(som, data):
    """Classifies each sample in data in one of the classes definited
    using the method labels_map.
    Returns a list of the same length of data where the i-th element
    is the class assigned to data[i].
    """
    winmap = som.labels_map(X_train, y_train)
    default_class = np.sum(list(winmap.values())).most_common()[0][0]
    result = []
    for d in data:
        win_position = som.winner(d)
        if win_position in winmap:
            result.append(winmap[win_position].most_common()[0][0])
        else:
            result.append(default_class)
    return result

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(data, labels, stratify=labels, test_size=0.1, random_state=42)

som = MiniSom(9, 9, data.shape[1], sigma=3, learning_rate=0.5, 
              neighborhood_function='triangle', random_seed=10)
som.pca_weights_init(X_train)
som.train_random(X_train, 500, verbose=False)

y_pred = classify(som, X_test)
report = classification_report(y_test, y_pred)
print(report)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# First plot: PCA projection colored by SOM clusters
scatter1 = ax1.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_index,
    cmap='RdYlGn',
    alpha=0.7,
    s=30
)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('PCA projection colored by SoM clusters', fontsize=14, pad=20)
plt.colorbar(scatter1, ax=ax1, label='SoM cluster')

# Second plot: Classification metrics
classes = ['AD/HD', 'C']
x_pos = np.arange(len(classes))
width = 0.25

colors = sns.color_palette("muted")

precision = [float(line.split()[1]) for line in report.split('\n')[2:4]]
recall = [float(line.split()[2]) for line in report.split('\n')[2:4]]
f1_score = [float(line.split()[3]) for line in report.split('\n')[2:4]] 

bars1 = ax2.bar(x_pos - width, precision, width, label='Precision', color=colors[0])
bars2 = ax2.bar(x_pos, recall, width, label='Recall', color=colors[1])
bars3 = ax2.bar(x_pos + width, f1_score, width, label='F1-Score', color=colors[2])

def add_labels(bars, ax):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

add_labels(bars1, ax2)
add_labels(bars2, ax2)
add_labels(bars3, ax2)

ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('SoM Classification Report', fontsize=14, pad=20)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(classes, fontsize=11)
ax2.set_ylim(0, 1)
# ax2.spines['top'].set_visible(False)
# ax2.spines['right'].set_visible(False)
ax2.legend(frameon=False, fontsize=11, loc='upper center', ncol=3,
          bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.show()